### Problem Statement

The data is originally taken from the [NYC Open Data website](https://opendata.cityofnewyork.us/) and contains data related to park events in the New York City area.

The data provided here contain two collections - **events** and **neighbourhoods**.

**events** collection documents have the following fields - 

- `event_id` - Unique event id

- `title` - Name of the event

- `start_date_time` - The start date and time of the event

- `end_date_time` - The end date and time of the event

- `snippet` - A brief description of the event

- `cost_free` - Indicating whether an event is free (0) or not (1)

- `must_see` - Indicates if event should be featured on Parks website with "Must See" banner. 0 if event is not featured and 1 if event is featured.

- `location_name` - Location name where event takes place

- `location` - Longitude and latitude of the location of event


**neighbourhoods** collection documents have the following fields -

- `properties` - Embedded document containing information related to the neighbourhood

>- `ntacode` - Neighbourhood code
>- `ntaname` - Neighbourhood name
>- `boro_code` - Code of borough in which neighbourhood falls
>- `boro_name` - name of borough in which neighbourhood falls

- `geometry` - GEOJSON object containing coordinates of boundary of the neighbourhood 



----

*The data for **events** collection has been originally taken from - https://data.cityofnewyork.us/browse?Data-Collection_Data-Collection=NYC+Parks+Events&sortBy=most_accessed&utf8=%E2%9C%93*

*The data for **neighbourhoods** collectio has been originally taken from - https://data.cityofnewyork.us/City-Government/Neighborhood-Tabulation-Areas-NTA-/cpf4-rkhq*


----

### Connecting to MongoDB


----

In [1]:
# Importing the required libraries
import pymongo
import pprint as pp

pp.sorted = lambda x, key=None: x

In [2]:
client = pymongo.MongoClient("mongodb://localhost:27017/")

---
### Importing data

----

In [3]:
# # Restore database
# !mongorestore /home/avadmin/Desktop/Mongo/Content/Indexing/Assignment/Data/indexing_assignment

In [4]:
# Database
db = client.Project

In [5]:
# List collections
db.list_collection_names()

['neighbourhoods', 'stock', 'event', 'Assignment']

In [6]:
# Sample document
pp.pprint(
    db.event.find_one()
)

{'_id': '60d9cb7310d0be7a7763857a',
 'event_id': 156947,
 'title': 'Birding Jamaica Bay',
 'start_date_time': '2017-12-23T13:00:00',
 'end_date_time': '2017-12-23T14:30:00',
 'snippet': 'Our Rangers will guide you t the best wildlife viewing spots in '
            'the urban jungle. Birding programs are appropriate for all skill '
            'levels and beginners are welcome.',
 'cost_free': 1,
 'must_see': 1,
 'location_name': 'Bay 32nd Street and Beach Channel Drive',
 'location': {'type': 'Point',
              'coordinates': [-73.76619720458984, 40.59870147705078]}}


In [7]:
# Sample document
pp.pprint(
    db.neighbourhoods.find_one()
)

{'_id': '60d9d8036fa8d9e558634f2e',
 'properties': {'ntacode': 'BK41',
                'ntaname': 'Kensington-Ocean Parkway',
                'boro_name': 'Brooklyn',
                'boro_code': '3'},
 'geometry': {'type': 'MultiPolygon',
              'coordinates': [[[[-73.97084113729679, 40.64637857107231],
                                [-73.97076976149238, 40.646110907343235],
                                [-73.9705455439065, 40.64531666812141],
                                [-73.9703167698547, 40.64453794134643],
                                [-73.97008897727095, 40.64374583950689],
                                [-73.96988464737747, 40.64303029756926],
                                [-73.96968611807544, 40.64235600977225],
                                [-73.96948468137924, 40.64168989653149],
                                [-73.969399031316, 40.64138244571584],
                                [-73.96891461058416, 40.63975715975412],
                                [

----
### Assignment Questions


Note - View all queries before attempting the questions. Use proper indexing to answer the questions.

----

### Q1

How many events were `must see events`?

In [8]:
db.event.count_documents ( { 'must_see': 1 } )

4360

### Q2

How `many events` were must see as well as `cost free`?

In [9]:
db.event.count_documents ( { 'must_see': 1, 'cost_free': 1 } )

3643

### Q3

How many `must see and cost free events` were held after `2018-01-01`?

In [10]:
from datetime import datetime

db.event.count_documents ( { 'must_see': 1, 'cost_free': 1, 'start_date_time': {'$gt': '2018-01-01T00:00:00' } } )

597

### Q4

How many indexes did you use to answer the above queries? List the index keys for each index used.

In [11]:
# For Q1
db.event.create_index ( 'must_see' )

'must_see_1'

In [12]:
# For Q2
db.event.create_index ( [ ( 'must_see', 1), ('cost_free', 1 ) ] )

'must_see_1_cost_free_1'

In [13]:
# For Q3
db.event.create_index ( [ ( 'must_see', 1), ('cost_free', 1), ('start_date_time', 1 ) ] )

'must_see_1_cost_free_1_start_date_time_1'

### Q5

What was the combined size of all the index created for the above queries?

In [14]:
db.event.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]},
 'must_see_1': {'v': 2, 'key': [('must_see', 1)]},
 'must_see_1_cost_free_1': {'v': 2,
  'key': [('must_see', 1), ('cost_free', 1)]},
 'must_see_1_cost_free_1_start_date_time_1': {'v': 2,
  'key': [('must_see', 1), ('cost_free', 1), ('start_date_time', 1)]},
 'title_text': {'v': 2,
  'key': [('_fts', 'text'), ('_ftsx', 1)],
  'weights': SON([('title', 1)]),
  'default_language': 'english',
  'language_override': 'language',
  'textIndexVersion': 3},
 'location_2dsphere': {'v': 2,
  'key': [('location', '2dsphere')],
  '2dsphereIndexVersion': 3}}

In [15]:
db.command ("collstats", "event") ["totalIndexSize"]

7540736

In [16]:
index_size = db.command ("collstats", "event") ["totalIndexSize"]
print(f"Total Index Size: {index_size / 1024:.2f} KB")

Total Index Size: 7364.00 KB


In [17]:
print(f"Total Index Size: {index_size / (1024*1024):.2f} MB")

Total Index Size: 7.19 MB


### Q6

How many events have the exact term `"Central Park" but not the term "Tour"` in the `title` field? 

***Hint - You will need to create a text index here.***

In [18]:
# Create Text Index on the "title" field

db.event.create_index ( [ ( 'title', 'text' ) ] )

'title_text'

In [19]:
# Query to Find Events with "Central Park" but not "Tour"

db.event.count_documents ( { '$text': {'$search': '"Central Park" -Tour' } } )

462

### Q7

How many events were held in `Williamsburg` neighbourhood of `Brooklyn` borough?

***Hint - Create geospatial index for this query. Use the `neighbourhoods` collection for geometry of the neighbourhood. Query on the `ntaname` and `boro_name` fields.***

In [20]:
# Create Geospatial Index on location field in event collection

db.event.create_index ( [ ( "location", "2dsphere" ) ] )

'location_2dsphere'

In [21]:
# Get the Boundary of Williamsburg from neighbourhoods collection

williamsburg_geom = db.neighbourhoods.find_one ( { "properties.ntaname": "Williamsburg", "properties.boro_name": "Brooklyn" } ) ["geometry"]

In [22]:
# Query to Get Events in Williamsburg

db.event.count_documents ( { "location": { "$geoWithin": { "$geometry": williamsburg_geom } } } )

5

### Q8

Name the title of the `paid and must see events` that are located maximum `500 meters` from the `Brooklyn Museum (coordinates = [-73.9636, 40.6712])` after `2018-06-06`.

In [23]:
# Create Geospatial Index on location field

db.event.create_index ( [ ( "location", "2dsphere" ) ] )

'location_2dsphere'

In [24]:
# Fetching the titles of the events
events = list(db.event.find ( {
    "cost_free": 1,
    "must_see": 1,
    "start_date_time": {"$gt": "2018-06-06T00:00:00"},
    "location": { "$nearSphere": { "$geometry": { "type": "Point", "coordinates": [-73.9636, 40.6712]  },
            
    "$maxDistance": 500 } } }, 
                             
    { "title": 1, "_id": 0 } ) )

# Printing the titles
for event in events:
    print(event['title'])

Brooklyn Chess Festival
